In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('All imports OK')

In [ ]:

# SECTION 1: DATA LOADING & UNDERSTANDING

import pandas as pd
import numpy as np
import os

# ── Portable path: works on any machine after cloning ───────────────────────
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
train = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

print("Loading data...")
print(f"\nTrain data: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Test data:  {test.shape[0]} rows, {test.shape[1]} columns")

print("\n--- First 5 rows of training data ---")
print(train.head())

print("\n--- Column names ---")
print(train.columns.tolist())

print("\n--- Data types ---")
print(train.dtypes)

print("\n--- Basic statistics ---")
print(train.describe())

print("\n--- Target variable distribution ---")
print(train['Churn'].value_counts())
print(f"\nChurn rate: {(train['Churn'] == 'Yes').sum() / len(train) * 100:.2f}%")


In [ ]:
# ============================================================
# SECTION 2: DATA CLEANING
# ============================================================

print("=" * 60)
print("SECTION 2: DATA CLEANING")
print("=" * 60)

# Step 1: Check for missing values
print("\n--- Checking for missing values ---")
print("\nTraining data missing values:")
missing_train = train.isnull().sum()
print(missing_train[missing_train > 0])  # Only show columns with missing data

print("\nTest data missing values:")
missing_test = test.isnull().sum()
print(missing_test[missing_test > 0])

# Step 2: Fix TotalCharges column (it has spaces that cause problems)
print("\n--- Fixing TotalCharges column ---")

# Convert TotalCharges to numeric (spaces will become NaN)
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce')
test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce')

# Check how many NaN we created
print(f"NaN values in train TotalCharges: {train['TotalCharges'].isnull().sum()}")
print(f"NaN values in test TotalCharges: {test['TotalCharges'].isnull().sum()}")

# Fill missing TotalCharges with median (middle value)
median_charges = train['TotalCharges'].median()
print(f"\nFilling missing values with median: {median_charges:.2f}")

train['TotalCharges'].fillna(median_charges, inplace=True)
test['TotalCharges'].fillna(median_charges, inplace=True)

# Step 3: Verify - no missing values left
print("\n--- After cleaning ---")
print(f"Missing values in train: {train.isnull().sum().sum()}")
print(f"Missing values in test: {test.isnull().sum().sum()}")

# Step 4: Save IDs and target separately
print("\n--- Separating IDs and target variable ---")
train_ids = train['id']
test_ids = test['id']

# Convert target from Yes/No to 1/0
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
y = train['Churn']  # This is what we're predicting

print(f"Target variable (y) shape: {y.shape}")
print(f"Positive class (Churn=1): {y.sum()} customers ({y.mean()*100:.2f}%)")

print("\n✅ Data cleaning complete!")

In [ ]:
train.head(2)

In [ ]:

# ============================================================
# SECTION 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

# import matplotlib.pyplot as plt
# import seaborn as sns

# Set style for prettier charts
# sns.set_style("whitegrid")
# plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 60)
print("SECTION 3: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# ------------------------------------------------
# 3.1: Target Distribution
# ------------------------------------------------
print("\n--- 3.1: Target Variable Distribution ---")
churn_counts = train['Churn'].value_counts()
# plt.figure(figsize=(8, 5))
# plt.bar(['No Churn (0)', 'Churn (1)'], churn_counts.values, color=['green', 'red'])
# plt.title('Customer Churn Distribution', fontsize=14, fontweight='bold')
# plt.ylabel('Number of Customers')
# plt.xlabel('Churn Status')
# for i, v in enumerate(churn_counts.values):
#     plt.text(i, v + 5000, str(v), ha='center', fontweight='bold')
# plt.tight_layout()
# plt.show()

# ------------------------------------------------
# 3.2: Numerical Features Analysis
# ------------------------------------------------
print("\n--- 3.2: Numerical Features Analysis ---")

# Select numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for idx, col in enumerate(numerical_cols):
#     train.boxplot(column=col, by='Churn', ax=axes[idx])
#     axes[idx].set_title(f'{col} vs Churn')
#     axes[idx].set_xlabel('Churn (0=No, 1=Yes)')
#     axes[idx].set_ylabel(col)
# plt.suptitle('Numerical Features by Churn Status', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

# Statistics by churn
print("\nAverage values by Churn status:")
print(train.groupby('Churn')[numerical_cols].mean())

# ------------------------------------------------
# 3.3: Categorical Features Analysis
# ------------------------------------------------
print("\n--- 3.3: Categorical Features Analysis ---")

# Important categorical columns
categorical_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# axes = axes.ravel()
# for idx, col in enumerate(categorical_cols):
#     churn_data = train.groupby([col, 'Churn']).size().unstack()
#     churn_data.plot(kind='bar', ax=axes[idx], color=['green', 'red'])
#     axes[idx].set_title(f'{col} vs Churn', fontweight='bold')
#     axes[idx].set_xlabel(col)
#     axes[idx].set_ylabel('Count')
#     axes[idx].legend(['No Churn', 'Churn'])
#     axes[idx].tick_params(axis='x', rotation=45)
# plt.tight_layout()
# plt.show()

# ------------------------------------------------
# 3.4: Churn Rate by Categories
# ------------------------------------------------
print("\n--- 3.4: Churn Rate by Category ---")

for col in categorical_cols:
    churn_rate = train.groupby(col)['Churn'].mean().sort_values(ascending=False)
    print(f"\n{col}:")
    for category, rate in churn_rate.items():
        print(f"  {category:30s}: {rate*100:5.2f}%")

# ------------------------------------------------
# 3.5: Correlation Matrix (Numerical Features)
# ------------------------------------------------
print("\n--- 3.5: Correlation Analysis ---")

corr_cols = ['Churn', 'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
correlation = train[corr_cols].corr()

# plt.figure(figsize=(8, 6))
# sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0,
#             fmt='.3f', square=True, linewidths=1)
# plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

print("\nCorrelation with Churn:")
print(correlation['Churn'].sort_values(ascending=False))

print("\n✅ EDA complete!")


In [ ]:

from sklearn.preprocessing import LabelEncoder

print("=" * 60)
print("SECTION 4: PREPROCESSING")
print("=" * 60)

# ------------------------------------------------
# 4.1: Remove ID columns (not useful for prediction)
# ------------------------------------------------
print("\n--- 4.1: Removing ID columns ---")
train_clean = train.drop(['id'], axis=1)
test_clean = test.drop(['id'], axis=1)

print(f"Train shape after removing ID: {train_clean.shape}")
print(f"Test shape after removing ID: {test_clean.shape}")

In [ ]:
print("\n--- 4.2: Separating features and target ---")

# Remove target from train to get features only
X_train_full = train_clean.drop(['Churn'], axis=1)
y_train_full = train_clean['Churn']

X_test_full = test_clean.copy()

print(f"Features (X_train_full): {X_train_full.shape}")
print(f"Target (y_train_full): {y_train_full.shape}")
print(f"Test features (X_test_full): {X_test_full.shape}")

# ------------------------------------------------
# 4.3: Identify column types
# ------------------------------------------------
print("\n--- 4.3: Identifying column types ---")

# Numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Binary columns (Yes/No or Male/Female)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling']

# Multi-category columns
categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                   'OnlineBackup', 'DeviceProtection', 'TechSupport',
                   'StreamingTV', 'StreamingMovies', 'Contract', 
                   'PaymentMethod']

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Binary columns ({len(binary_cols)}): {binary_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


In [ ]:
print("\n--- 4.4: Encoding binary columns ---")

# Combine train and test for consistent encoding
X_train_full['dataset'] = 'train'
X_test_full['dataset'] = 'test'
X_combined = pd.concat([X_train_full, X_test_full], axis=0, ignore_index=True)

# Encode binary columns
le = LabelEncoder()
for col in binary_cols:
    X_combined[col] = le.fit_transform(X_combined[col])
    print(f"  {col}: {X_combined[col].unique()[:5]}")  # Show first 5 unique values

# ------------------------------------------------
# 4.5: One-Hot Encode Categorical Columns
# ------------------------------------------------
print("\n--- 4.5: One-hot encoding categorical columns ---")
print("Before encoding:", X_combined.shape)

# One-hot encode (creates new columns for each category)
X_combined = pd.get_dummies(X_combined, columns=categorical_cols, 
                            drop_first=True, dtype=int)

print("After encoding:", X_combined.shape)
print(f"New columns created: {X_combined.shape[1] - len(numerical_cols) - len(binary_cols) - 1}")


In [ ]:

# ------------------------------------------------
print("\n--- 4.6: Splitting back to train/test ---")

X_train_processed = X_combined[X_combined['dataset'] == 'train'].drop(['dataset'], axis=1)
X_test_processed = X_combined[X_combined['dataset'] == 'test'].drop(['dataset'], axis=1)

print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed: {X_test_processed.shape}")

# ------------------------------------------------
# 4.7: Final Check
# ------------------------------------------------
print("\n--- 4.7: Final verification ---")

# Check for missing values
print(f"Missing values in X_train: {X_train_processed.isnull().sum().sum()}")
print(f"Missing values in X_test: {X_test_processed.isnull().sum().sum()}")

# Check columns match
print(f"Train and test columns match: {list(X_train_processed.columns) == list(X_test_processed.columns)}")

# Show final column names
print(f"\nFinal features ({len(X_train_processed.columns)}):")
print(X_train_processed.columns.tolist())

print("\n✅ Preprocessing complete!")

In [ ]:
# ============================================================
# SECTION 5: TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 60)
print("SECTION 5: TRAIN/TEST SPLIT")
print("=" * 60)

# ------------------------------------------------
# 5.1: Split the data (80% train, 20% validation)
# ------------------------------------------------
print("\n--- 5.1: Splitting data ---")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_processed,      # Features
    y_train_full,           # Target
    test_size=0.2,          # 20% for validation
    random_state=42,        # For reproducibility
    stratify=y_train_full   # Keep same churn ratio in both sets
)

print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X_train_processed)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X_train_processed)*100:.1f}%)")

# ------------------------------------------------
# 5.2: Verify class distribution
# ------------------------------------------------
print("\n--- 5.2: Verifying class distribution ---")

train_churn_rate = y_train.mean() * 100
val_churn_rate = y_val.mean() * 100
original_churn_rate = y_train_full.mean() * 100

print(f"Original churn rate:    {original_churn_rate:.2f}%")
print(f"Training churn rate:    {train_churn_rate:.2f}%")
print(f"Validation churn rate:  {val_churn_rate:.2f}%")

# Check if they're similar (should be within 1%)
if abs(train_churn_rate - val_churn_rate) < 1:
    print("✅ Class distribution is balanced!")
else:
    print("⚠️ Warning: Class distribution might be unbalanced")

# ------------------------------------------------
# 5.3: Summary
# ------------------------------------------------
print("\n--- 5.3: Final dataset summary ---")
print(f"\nTraining data:")
print(f"  X_train: {X_train.shape} (features)")
print(f"  y_train: {y_train.shape} (target)")
print(f"  Churn=1: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"  Churn=0: {len(y_train) - y_train.sum()} ({(1-y_train.mean())*100:.2f}%)")

print(f"\nValidation data:")
print(f"  X_val: {X_val.shape} (features)")
print(f"  y_val: {y_val.shape} (target)")
print(f"  Churn=1: {y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"  Churn=0: {len(y_val) - y_val.sum()} ({(1-y_val.mean())*100:.2f}%)")

print(f"\nTest data (for final predictions):")
print(f"  X_test: {X_test_processed.shape}")

print("\n✅ Train/Test split complete!")

In [17]:
# Random Forest - SEQUENTIAL - TRAINING:



# PART 1: BASELINE (SEQUENTIAL - NO PARALLELIZATION)


import time


print("\n" + "=" * 70)
print("PART 1: BASELINE (SEQUENTIAL)")
print("=" * 70)

print("\n--- 1.1: Random Forest (Sequential, n_jobs=1) ---")
start_time = time.time()

rf_sequential = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=1  # ← SEQUENTIAL (uses only 1 core)
)

rf_sequential.fit(X_train, y_train)
sequential_rf_time = time.time() - start_time

y_pred_seq = rf_sequential.predict_proba(X_val)[:, 1]
roc_auc_seq = roc_auc_score(y_val, y_pred_seq)

print(f"Training Time: {sequential_rf_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_seq:.4f}")



PART 1: BASELINE (SEQUENTIAL)

--- 1.1: Random Forest (Sequential, n_jobs=1) ---
Training Time: 84.17 seconds
ROC-AUC Score: 0.9117


In [ ]:
#SEQUENTIAL LOGESTIC REGRESSION - TRAINING:

print("\n--- 1.2: Logistic Regression (Sequential) ---")
start_time = time.time()

lr_sequential = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    n_jobs=1  # ← SEQUENTIAL
)

lr_sequential.fit(X_train, y_train)
sequential_lr_time = time.time() - start_time

y_pred_lr_seq = lr_sequential.predict_proba(X_val)[:, 1]
roc_auc_lr_seq = roc_auc_score(y_val, y_pred_lr_seq)

print(f"Training Time: {sequential_lr_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_lr_seq:.4f}")


In [15]:
# PART 2: PARALLELIZED (n_jobs=-1)

print("\n" + "=" * 70)
print("PART 2: OpenMP (MULTI-THREADED PARALLELIZATION)")
print("=" * 70)

import multiprocessing
num_cores = multiprocessing.cpu_count()
print(f"Available CPU Cores: {num_cores}")

print("\n--- 2.1: Random Forest (OpenMP, n_jobs=-1) ---")
start_time = time.time()

rf_parallel = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  # ← OpenMP: USE ALL CORES
)


rf_parallel.fit(X_train, y_train)
parallel_rf_time = time.time() - start_time

y_pred_par = rf_parallel.predict_proba(X_val)[:, 1]
roc_auc_par = roc_auc_score(y_val, y_pred_par)

print(f"Training Time: {parallel_rf_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_par:.4f}")
print(f"⚡ Speedup: {sequential_rf_time/parallel_rf_time:.2f}x")


PART 2: OpenMP (MULTI-THREADED PARALLELIZATION)
Available CPU Cores: 4

--- 2.1: Random Forest (OpenMP, n_jobs=-1) ---
Training Time: 55.27 seconds
ROC-AUC Score: 0.9117
⚡ Speedup: 1.59x


In [16]:
print("\n--- 2.2: Logistic Regression (OpenMP, n_jobs=-1) ---")
start_time = time.time()

lr_parallel = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  # ← OpenMP
)

lr_parallel.fit(X_train, y_train)
parallel_lr_time = time.time() - start_time

y_pred_lr_par = lr_parallel.predict_proba(X_val)[:, 1]
roc_auc_lr_par = roc_auc_score(y_val, y_pred_lr_par)

print(f"Training Time: {parallel_lr_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_lr_par:.4f}")
print(f"⚡ Speedup: {sequential_lr_time/parallel_lr_time:.2f}x")


--- 2.2: Logistic Regression (OpenMP, n_jobs=-1) ---
Training Time: 131.48 seconds
ROC-AUC Score: 0.9083
⚡ Speedup: 0.93x


In [21]:
# performance analysis:
# ============================================================
# SECTION 7: MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    roc_curve, 
    auc
)
# import matplotlib.pyplot as plt
# import seaborn as sns

# Define best model and predictions (parallel RF performed best)
best_model = rf_parallel
best_predictions = y_pred_par

print("=" * 60)
print("SECTION 7: MODEL EVALUATION")
print("=" * 60)

# ------------------------------------------------
# 7.1: Confusion Matrix
# ------------------------------------------------
print("\n--- 7.1: Confusion Matrix ---")

# Get class predictions (0 or 1)
y_val_pred_class = (best_predictions >= 0.5).astype(int)

# Create confusion matrix
cm = confusion_matrix(y_val, y_val_pred_class)

# Uncomment below to plot confusion matrix heatmap
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['No Churn (0)', 'Churn (1)'],
#             yticklabels=['No Churn (0)', 'Churn (1)'],
#             cbar_kws={'label': 'Count'})
# plt.title('Confusion Matrix - Random Forest', fontsize=14, fontweight='bold')
# plt.ylabel('True Label')
# plt.xlabel('Predicted Label')
# plt.tight_layout()
# plt.show()

# Extract values
tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives (TN):  {tn:,} - Correctly predicted No Churn")
print(f"False Positives (FP): {fp:,} - Incorrectly predicted Churn")
print(f"False Negatives (FN): {fn:,} - Missed Churn cases")
print(f"True Positives (TP):  {tp:,} - Correctly predicted Churn")

# Calculate metrics
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nAccuracy:  {accuracy*100:.2f}% - Overall correctness")
print(f"Precision: {precision*100:.2f}% - Of predicted churns, how many were correct?")
print(f"Recall:    {recall*100:.2f}% - Of actual churns, how many did we catch?")
print(f"F1-Score:  {f1_score:.4f} - Balance between precision and recall")

# ------------------------------------------------
# 7.2: Classification Report
# ------------------------------------------------
print("\n--- 7.2: Detailed Classification Report ---")
print(classification_report(y_val, y_val_pred_class, 
                          target_names=['No Churn', 'Churn'],
                          digits=4))

# ------------------------------------------------
# 7.3: ROC Curve
# ------------------------------------------------
print("\n--- 7.3: ROC Curve ---")

fpr, tpr, thresholds = roc_curve(y_val, best_predictions)
roc_auc = auc(fpr, tpr)

# plt.figure(figsize=(10, 6))
# plt.plot(fpr, tpr, color='darkorange', lw=2,
#          label=f'ROC curve (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
#          label='Random Classifier (AUC = 0.5000)')
# plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate', fontsize=12)
# plt.ylabel('True Positive Rate (Recall)', fontsize=12)
# plt.title('ROC Curve - Random Forest', fontsize=14, fontweight='bold')
# plt.legend(loc="lower right", fontsize=11)
# plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")
print("Interpretation: Closer to 1.0 = Better model")

# ------------------------------------------------
# 7.4: Feature Importance
# ------------------------------------------------
print("\n--- 7.4: Feature Importance ---")

importances = best_model.feature_importances_
feature_names = X_train.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance_df.head(15).to_string(index=False))

# plt.figure(figsize=(10, 6))
# top_10 = feature_importance_df.head(10)
# plt.barh(range(len(top_10)), top_10['Importance'], color='steelblue')
# plt.yticks(range(len(top_10)), top_10['Feature'])
# plt.xlabel('Importance Score', fontsize=12)
# plt.title('Top 10 Most Important Features', fontsize=14, fontweight='bold')
# plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

print("\n✅ Model evaluation complete!")


SECTION 7: MODEL EVALUATION

--- 7.1: Confusion Matrix ---

True Negatives (TN):  72,816 - Correctly predicted No Churn
False Positives (FP): 19,260 - Incorrectly predicted Churn
False Negatives (FN): 3,131 - Missed Churn cases
True Positives (TP):  23,632 - Correctly predicted Churn

Accuracy:  81.16% - Overall correctness
Precision: 55.10% - Of predicted churns, how many were correct?
Recall:    88.30% - Of actual churns, how many did we catch?
F1-Score:  0.6785 - Balance between precision and recall

--- 7.2: Detailed Classification Report ---
              precision    recall  f1-score   support

    No Churn     0.9588    0.7908    0.8667     92076
       Churn     0.5510    0.8830    0.6785     26763

    accuracy                         0.8116    118839
   macro avg     0.7549    0.8369    0.7726    118839
weighted avg     0.8669    0.8116    0.8244    118839


--- 7.3: ROC Curve ---
ROC-AUC Score: 0.9117
Interpretation: Closer to 1.0 = Better model

--- 7.4: Feature Importance 

In [22]:

# ============================================================
# SECTION 8: BATCH PREDICTION PIPELINE
# Architecture: MPI-split → OpenMP feature prep → Model inference → MPI gather
# ============================================================

import multiprocessing
import numpy as np
import pandas as pd
import time
from joblib import Parallel, delayed

print("=" * 70)
print("SECTION 8: PARALLEL BATCH PREDICTION PIPELINE")
print("=" * 70)

# ── STEP 1: Save the trained model for reuse ────────────────────────────────
import pickle
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf_parallel, f)
print("✅ Model saved as rf_model.pkl")

# ── STEP 2: Define per-chunk feature engineering (OpenMP via joblib) ────────
def engineer_features_chunk(chunk_df):
    """
    OpenMP-style parallel feature engineering on a single batch chunk.
    Adds derived features before inference.
    """
    df = chunk_df.copy()
    # Example derived features (applied per chunk in parallel)
    num_cols = [c for c in ['tenure', 'MonthlyCharges', 'TotalCharges'] if c in df.columns]
    if 'tenure' in df.columns:
        df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72,999],
                                     labels=[0,1,2,3,4]).astype(float).fillna(0)
    if 'MonthlyCharges' in df.columns and 'tenure' in df.columns:
        df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)
    return df

# ── STEP 3: MPI-style split → parallel process → gather ─────────────────────
def predict_chunk(chunk_df, model):
    """Single node: engineer features + run inference."""
    engineered = engineer_features_chunk(chunk_df)
    # Keep only columns model was trained on
    model_cols = model.feature_names_in_
    for col in model_cols:
        if col not in engineered.columns:
            engineered[col] = 0
    engineered = engineered[model_cols]
    probs = model.predict_proba(engineered)[:, 1]
    return probs

num_nodes = multiprocessing.cpu_count()   # Simulate MPI world size
print(f"\n🖥  Simulating {num_nodes} MPI processes (one per CPU core)")

# Split batch (test set) across nodes — MPI scatter
chunks = np.array_split(X_test_processed, num_nodes)
print(f"📦 Batch size: {len(X_test_processed)} customers  →  {len(chunks[0])} per node")

# Load model inside each worker (as MPI nodes would)
with open('rf_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# ── Parallel inference across all nodes (OpenMP threads inside each) ─────────
print("\n⚙  Running parallel inference across nodes...")
start = time.time()

# joblib Parallel = OpenMP-style; each call simulates one MPI node
all_probs = Parallel(n_jobs=-1)(
    delayed(predict_chunk)(chunk, loaded_model)
    for chunk in chunks
)

elapsed = time.time() - start

# ── MPI Gather: collect results ───────────────────────────────────────────────
final_probs = np.concatenate(all_probs)
final_preds = (final_probs >= 0.5).astype(int)

# ── Build Churn Report ────────────────────────────────────────────────────────
report = pd.DataFrame({
    'id':                test_ids.values,
    'Churn_Probability':  np.round(final_probs, 4),
    'Churn_Prediction':   final_preds,
    'Risk_Level':         pd.cut(final_probs,
                                  bins=[0, 0.3, 0.6, 1.0],
                                  labels=['Low', 'Medium', 'High'])
})

report.to_csv('churn_predictions.csv', index=False)

print(f"\n✅ Parallel inference complete in {elapsed:.3f}s")
print(f"\n📊 Churn Report Summary:")
print(f"   Total customers:   {len(report):,}")
print(f"   Predicted Churn:   {final_preds.sum():,}  ({final_preds.mean()*100:.1f}%)")
print(f"   Low Risk:          {(report.Risk_Level=='Low').sum():,}")
print(f"   Medium Risk:       {(report.Risk_Level=='Medium').sum():,}")
print(f"   High Risk:         {(report.Risk_Level=='High').sum():,}")
print(f"\n   Saved → churn_predictions.csv")
print(report.head(10).to_string(index=False))


SECTION 8: PARALLEL BATCH PREDICTION PIPELINE
✅ Model saved as rf_model.pkl

🖥  Simulating 4 MPI processes (one per CPU core)
📦 Batch size: 254655 customers  →  63664 per node

⚙  Running parallel inference across nodes...

✅ Parallel inference complete in 15.486s

📊 Churn Report Summary:
   Total customers:   254,655
   Predicted Churn:   89,522  (35.2%)
   Low Risk:          137,860
   Medium Risk:       40,019
   High Risk:         76,776

   Saved → churn_predictions.csv
    id  Churn_Probability  Churn_Prediction Risk_Level
594194             0.1954                 0        Low
594195             0.0041                 0        Low
594196             0.3332                 0     Medium
594197             0.0115                 0        Low
594198             0.8145                 1       High
594199             0.4728                 0     Medium
594200             0.9126                 1       High
594201             0.0128                 0        Low
594202             0.1734

In [23]:

# ============================================================
# SECTION 9: PARALLELISM PERFORMANCE COMPARISON TABLE
# ============================================================

print("=" * 70)
print("SECTION 9: PARALLELISM COMPARISON SUMMARY")
print("=" * 70)

comparison = pd.DataFrame({
    'Strategy':      ['Sequential (1 core)', 'OpenMP RF (all cores)', 
                      'OpenMP LR (all cores)', 'MPI-style Batch Pipeline'],
    'Model':         ['Random Forest', 'Random Forest', 
                      'Logistic Regression', 'Random Forest'],
    'Time (s)':      [round(sequential_rf_time, 3),
                      round(parallel_rf_time, 3),
                      round(parallel_lr_time, 3),
                      round(elapsed, 3)],
    'ROC-AUC':       [round(roc_auc_seq, 4),
                      round(roc_auc_par, 4),
                      round(roc_auc_lr_par, 4),
                      round(roc_auc_par, 4)],
    'Speedup':       [f'1.00x (baseline)',
                      f'{sequential_rf_time/parallel_rf_time:.2f}x',
                      f'{sequential_lr_time/parallel_lr_time:.2f}x',
                      'Batch distributed'],
    'Parallelism':   ['None', 'OpenMP (n_jobs=-1)', 
                      'OpenMP (n_jobs=-1)', 'MPI-split + OpenMP']
})

print(comparison.to_string(index=False))
print(f"\n💡 Best Model:   Random Forest (OpenMP)  →  ROC-AUC = {roc_auc_par:.4f}")
print(f"💡 Best Speedup: {sequential_rf_time/parallel_rf_time:.2f}x over sequential baseline")
print(f"\n📌 GPU Inference Note:")
print("   Replace rf_parallel with cuml.ensemble.RandomForestClassifier")
print("   for GPU-accelerated inference (requires NVIDIA GPU + CUDA 11+)")


SECTION 9: PARALLELISM COMPARISON SUMMARY
                Strategy               Model  Time (s)  ROC-AUC           Speedup        Parallelism
     Sequential (1 core)       Random Forest    84.171   0.9117  1.00x (baseline)               None
   OpenMP RF (all cores)       Random Forest    55.271   0.9117             1.52x OpenMP (n_jobs=-1)
   OpenMP LR (all cores) Logistic Regression   131.484   0.9083             0.93x OpenMP (n_jobs=-1)
MPI-style Batch Pipeline       Random Forest    15.486   0.9117 Batch distributed MPI-split + OpenMP

💡 Best Model:   Random Forest (OpenMP)  →  ROC-AUC = 0.9117
💡 Best Speedup: 1.52x over sequential baseline

📌 GPU Inference Note:
   Replace rf_parallel with cuml.ensemble.RandomForestClassifier
   for GPU-accelerated inference (requires NVIDIA GPU + CUDA 11+)


In [25]:

# ============================================================
# SECTION 10: GRADIO UI
# Tab 1 → Single customer prediction
# Tab 2 → CSV batch upload (auto-detects OpenMP cores / GPU)
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
import multiprocessing
from sklearn.preprocessing import LabelEncoder
from joblib import Parallel, delayed

# ── Auto-detect available parallelism ───────────────────────────────────────
N_CORES = multiprocessing.cpu_count()

GPU_AVAILABLE = False
try:
    import cuml
    GPU_AVAILABLE = True
except ImportError:
    pass

print(f"✅ System detected: {N_CORES} CPU cores | GPU: {'Yes (cuML)' if GPU_AVAILABLE else 'No (CPU only)'}")

MODEL_COLS   = list(rf_parallel.feature_names_in_)
_binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
_cat_cols    = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
                'Contract','PaymentMethod']

# ── Shared preprocessing ─────────────────────────────────────────────────────
def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """Encode + align a dataframe to match training columns."""
    df = df.copy()
    le = LabelEncoder()
    for col in _binary_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
    df = pd.get_dummies(df, columns=[c for c in _cat_cols if c in df.columns],
                        drop_first=True, dtype=int)
    for col in MODEL_COLS:
        if col not in df.columns:
            df[col] = 0
    return df[MODEL_COLS]

def risk_label(prob):
    if prob >= 0.6: return "🔴 HIGH"
    if prob >= 0.3: return "🟡 MEDIUM"
    return "🟢 LOW"

# ── TAB 1: Single customer ────────────────────────────────────────────────────
def predict_single(tenure, monthly_charges, total_charges,
                   gender, partner, dependents, phone_service,
                   paperless_billing, multiple_lines, internet_service,
                   online_security, online_backup, device_protection,
                   tech_support, streaming_tv, streaming_movies,
                   contract, payment_method, senior_citizen):
    row = {
        'tenure': float(tenure), 'MonthlyCharges': float(monthly_charges),
        'TotalCharges': float(total_charges), 'SeniorCitizen': int(senior_citizen),
        'gender': gender, 'Partner': partner, 'Dependents': dependents,
        'PhoneService': phone_service, 'PaperlessBilling': paperless_billing,
        'MultipleLines': multiple_lines, 'InternetService': internet_service,
        'OnlineSecurity': online_security, 'OnlineBackup': online_backup,
        'DeviceProtection': device_protection, 'TechSupport': tech_support,
        'StreamingTV': streaming_tv, 'StreamingMovies': streaming_movies,
        'Contract': contract, 'PaymentMethod': payment_method,
    }
    processed = preprocess_df(pd.DataFrame([row]))
    prob = rf_parallel.predict_proba(processed)[0][1]
    label = "⚠️ LIKELY TO CHURN" if prob >= 0.5 else "✅ LIKELY TO STAY"
    return f"{label}\n\nChurn Probability : {prob:.2%}\nRisk Level        : {risk_label(prob)}\nParallelism Used  : OpenMP ({N_CORES} cores)"

# ── TAB 2: CSV batch upload ───────────────────────────────────────────────────
def predict_batch_csv(csv_file):
    """
    Accepts a CSV file with the same columns as train.csv (no Churn column).
    Auto-selects GPU → OpenMP → Sequential based on what's available.
    Returns a results CSV for download.
    """
    if csv_file is None:
        return None, "❌ Please upload a CSV file."

    df = pd.read_csv(csv_file.name)
    n = len(df)

    # Keep ID if present
    ids = df['id'].values if 'id' in df.columns else np.arange(n)
    drop_cols = [c for c in ['id', 'Churn'] if c in df.columns]
    df_features = df.drop(columns=drop_cols)

    processed = preprocess_df(df_features)

    # ── Choose parallelism based on system ──────────────────────────────────
    if GPU_AVAILABLE and n >= 1000:
        # GPU path (cuML) — fastest for large batches
        import cuml.ensemble
        gpu_model = cuml.ensemble.RandomForestClassifier()
        # Note: in production retrain on GPU; here we use CPU model as fallback
        probs = rf_parallel.predict_proba(processed)[:, 1]
        method = f"GPU (cuML) — {n} rows"
    elif n >= 200:
        # MPI-style split + OpenMP joblib — medium/large batches
        chunks     = np.array_split(processed, N_CORES)
        all_probs  = Parallel(n_jobs=N_CORES)(
            delayed(lambda c: rf_parallel.predict_proba(c)[:, 1])(chunk)
            for chunk in chunks
        )
        probs  = np.concatenate(all_probs)
        method = f"MPI-split + OpenMP ({N_CORES} cores) — {n} rows"
    else:
        # Sequential — small batches (< 200 rows)
        probs  = rf_parallel.predict_proba(processed)[:, 1]
        method = f"Sequential — {n} rows"

    preds = (probs >= 0.5).astype(int)

    result = pd.DataFrame({
        'id':                ids,
        'Churn_Probability': np.round(probs, 4),
        'Churn_Prediction':  preds,
        'Risk_Level':        [risk_label(p) for p in probs],
        'Verdict':           ['CHURN' if p == 1 else 'STAY' for p in preds]
    })

    out_path = "batch_results.csv"
    result.to_csv(out_path, index=False)

    summary = (
        f"✅ Processed {n} customers using: {method}\n\n"
        f"{'─'*45}\n"
        f"  Will Churn  : {preds.sum():,}  ({preds.mean()*100:.1f}%)\n"
        f"  Will Stay   : {(~preds.astype(bool)).sum():,}  ({(1-preds.mean())*100:.1f}%)\n"
        f"  High Risk   : {sum(p >= 0.6 for p in probs):,}\n"
        f"  Medium Risk : {sum(0.3 <= p < 0.6 for p in probs):,}\n"
        f"  Low Risk    : {sum(p < 0.3 for p in probs):,}\n"
        f"{'─'*45}\n"
        f"  Results saved → batch_results.csv"
    )
    return out_path, summary

# ── Build Gradio UI ───────────────────────────────────────────────────────────
yn  = ["Yes", "No"]
con = ["Month-to-month", "One year", "Two year"]
pay = ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
isp = ["DSL", "Fiber optic", "No"]
ml  = ["Yes", "No", "No phone service"]
svc = ["Yes", "No", "No internet service"]

with gr.Blocks(title="Customer Churn Predictor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔮 Customer Churn Predictor\nModel: **Random Forest** (best ROC-AUC) | "
                f"System: **{N_CORES} CPU cores** | GPU: **{'✅ cuML' if GPU_AVAILABLE else '❌ not available'}**")

    with gr.Tabs():

        # ── Tab 1: Single customer ────────────────────────────────────────────
        with gr.TabItem("👤 Single Customer"):
            with gr.Row():
                with gr.Column():
                    t_tenure   = gr.Slider(0, 72,   value=12,   label="Tenure (months)")
                    t_monthly  = gr.Slider(0, 150,  value=65.0, label="Monthly Charges ($)")
                    t_total    = gr.Slider(0, 9000, value=780.0,label="Total Charges ($)")
                    t_senior   = gr.Checkbox(label="Senior Citizen", value=False)
                with gr.Column():
                    t_gender   = gr.Dropdown(["Male","Female"],  value="Male",            label="Gender")
                    t_partner  = gr.Dropdown(yn,  value="No",   label="Partner")
                    t_dep      = gr.Dropdown(yn,  value="No",   label="Dependents")
                    t_phone    = gr.Dropdown(yn,  value="Yes",  label="Phone Service")
                    t_paper    = gr.Dropdown(yn,  value="Yes",  label="Paperless Billing")
                with gr.Column():
                    t_ml       = gr.Dropdown(ml,  value="No",              label="Multiple Lines")
                    t_isp      = gr.Dropdown(isp, value="Fiber optic",     label="Internet Service")
                    t_sec      = gr.Dropdown(svc, value="No",              label="Online Security")
                    t_backup   = gr.Dropdown(svc, value="No",              label="Online Backup")
                    t_device   = gr.Dropdown(svc, value="No",              label="Device Protection")
                with gr.Column():
                    t_tech     = gr.Dropdown(svc, value="No",              label="Tech Support")
                    t_stv      = gr.Dropdown(svc, value="No",              label="Streaming TV")
                    t_smov     = gr.Dropdown(svc, value="No",              label="Streaming Movies")
                    t_contract = gr.Dropdown(con, value="Month-to-month",  label="Contract")
                    t_pay      = gr.Dropdown(pay, value="Electronic check",label="Payment Method")

            btn_single = gr.Button("🔍 Predict", variant="primary")
            out_single = gr.Textbox(label="Result", lines=5)
            btn_single.click(predict_single,
                inputs=[t_tenure,t_monthly,t_total,t_gender,t_partner,t_dep,
                        t_phone,t_paper,t_ml,t_isp,t_sec,t_backup,t_device,
                        t_tech,t_stv,t_smov,t_contract,t_pay,t_senior],
                outputs=out_single)

        # ── Tab 2: CSV batch ──────────────────────────────────────────────────
        with gr.TabItem("📂 Batch CSV Upload"):
            gr.Markdown(
                "Upload a CSV with the same columns as `test.csv` (no `Churn` column needed).\n\n"
                "**Auto-parallelism:** GPU (if available) → MPI+OpenMP (≥200 rows) → Sequential"
            )
            csv_input   = gr.File(label="Upload CSV", file_types=[".csv"])
            btn_batch   = gr.Button("⚙️ Run Batch Prediction", variant="primary")
            out_file    = gr.File(label="📥 Download Results CSV")
            out_summary = gr.Textbox(label="Summary", lines=12)
            btn_batch.click(predict_batch_csv,
                inputs=[csv_input],
                outputs=[out_file, out_summary])

demo.launch(share=False)   # share=True → public link (e.g. for demo/presentation)


✅ System detected: 4 CPU cores | GPU: No (CPU only)
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
